In [2]:
import random
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd


# ---------------------------------------------------------
# 1. Project settings
# ---------------------------------------------------------

random.seed(42)

start_date = datetime(2025, 1, 1)
end_date = datetime(2026, 7, 1)  # Exclusive: data runs through June 30, 2026.

number_of_visitors = 7000
number_of_sessions = 45000

# Customer and order counts are allowed to vary with generated behaviour.


# ---------------------------------------------------------
# 2. Output folder
# ---------------------------------------------------------

# This works when the script is run from the project root, src, or notebooks.
project_dir = Path.cwd()
if project_dir.name in ["src", "notebooks"]:
    project_dir = project_dir.parent

raw_dir = project_dir / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

print("Raw data folder:", raw_dir)
print("Existing CSVs with matching names will be overwritten.")


# ---------------------------------------------------------
# 3. Campaign data and internal campaign behaviour
# ---------------------------------------------------------

# Public campaign columns stay the same. The extra profile values below are
# used only to make the generated traffic, conversions, and spend less uniform.
channel_details = [
    {
        "channel": "Paid Search",
        "campaign_type": "Paid",
        "cost_tracking_available": True,
        "campaign_count": 3,
        "traffic_weight": 1.20,
        "cart_multiplier": 1.08,
        "checkout_multiplier": 1.10,
        "purchase_multiplier": 1.18,
        "repeat_multiplier": 1.00,
        "order_value_multiplier": 1.08,
        "discount_multiplier": 0.90,
        "refund_multiplier": 1.00,
        "daily_impressions": 2600,
        "ctr": 0.045,
        "cpc": 1.55,
    },
    {
        "channel": "Paid Social",
        "campaign_type": "Paid",
        "cost_tracking_available": True,
        "campaign_count": 3,
        "traffic_weight": 1.40,
        "cart_multiplier": 0.94,
        "checkout_multiplier": 0.91,
        "purchase_multiplier": 0.86,
        "repeat_multiplier": 0.88,
        "order_value_multiplier": 0.96,
        "discount_multiplier": 1.25,
        "refund_multiplier": 1.10,
        "daily_impressions": 4300,
        "ctr": 0.022,
        "cpc": 0.82,
    },
    {
        "channel": "Display",
        "campaign_type": "Paid",
        "cost_tracking_available": True,
        "campaign_count": 3,
        "traffic_weight": 1.12,
        "cart_multiplier": 0.78,
        "checkout_multiplier": 0.82,
        "purchase_multiplier": 0.72,
        "repeat_multiplier": 0.80,
        "order_value_multiplier": 0.91,
        "discount_multiplier": 1.12,
        "refund_multiplier": 1.12,
        "daily_impressions": 6800,
        "ctr": 0.008,
        "cpc": 0.38,
    },
    {
        "channel": "Email",
        "campaign_type": "Owned",
        "cost_tracking_available": True,
        "campaign_count": 3,
        "traffic_weight": 0.76,
        "cart_multiplier": 1.18,
        "checkout_multiplier": 1.20,
        "purchase_multiplier": 1.30,
        "repeat_multiplier": 1.35,
        "order_value_multiplier": 1.05,
        "discount_multiplier": 1.08,
        "refund_multiplier": 0.82,
        "daily_impressions": 1500,
        "ctr": 0.065,
        "cpc": 0.24,
    },
    {
        "channel": "Organic Search",
        "campaign_type": "Organic",
        "cost_tracking_available": False,
        "campaign_count": 3,
        "traffic_weight": 1.28,
        "cart_multiplier": 1.09,
        "checkout_multiplier": 1.08,
        "purchase_multiplier": 1.12,
        "repeat_multiplier": 1.04,
        "order_value_multiplier": 1.07,
        "discount_multiplier": 0.94,
        "refund_multiplier": 0.92,
        "daily_impressions": None,
        "ctr": None,
        "cpc": None,
    },
    {
        "channel": "Referral",
        "campaign_type": "Referral",
        "cost_tracking_available": False,
        "campaign_count": 3,
        "traffic_weight": 0.68,
        "cart_multiplier": 1.05,
        "checkout_multiplier": 1.05,
        "purchase_multiplier": 1.13,
        "repeat_multiplier": 1.08,
        "order_value_multiplier": 1.00,
        "discount_multiplier": 0.92,
        "refund_multiplier": 0.92,
        "daily_impressions": None,
        "ctr": None,
        "cpc": None,
    },
    {
        "channel": "Direct",
        "campaign_type": "Direct",
        "cost_tracking_available": False,
        "campaign_count": 1,
        "traffic_weight": 1.00,
        "cart_multiplier": 1.00,
        "checkout_multiplier": 1.05,
        "purchase_multiplier": 1.12,
        "repeat_multiplier": 1.04,
        "order_value_multiplier": 1.06,
        "discount_multiplier": 0.86,
        "refund_multiplier": 0.86,
        "daily_impressions": None,
        "ctr": None,
        "cpc": None,
    },
]

campaigns = []
campaign_profiles = {}

for detail in channel_details:
    for campaign_number in range(1, detail["campaign_count"] + 1):
        campaign_id = f"CMP{len(campaigns) + 1:03d}"

        if detail["channel"] == "Direct":
            campaign_name = "Direct Traffic"
        else:
            campaign_name = (
                f"{detail['channel']} Campaign {campaign_number}"
            )

        campaigns.append({
            "campaign_id": campaign_id,
            "campaign_name": campaign_name,
            "channel": detail["channel"],
            "campaign_type": detail["campaign_type"],
            "cost_tracking_available": detail["cost_tracking_available"],
        })

        # Each campaign also gets its own performance range within its channel.
        campaign_profiles[campaign_id] = {
            "channel": detail["channel"],
            "traffic_weight": detail["traffic_weight"] * random.uniform(0.75, 1.25),
            "cart_multiplier": detail["cart_multiplier"] * random.uniform(0.88, 1.12),
            "checkout_multiplier": detail["checkout_multiplier"] * random.uniform(0.88, 1.12),
            "purchase_multiplier": detail["purchase_multiplier"] * random.uniform(0.82, 1.18),
            "repeat_multiplier": detail["repeat_multiplier"] * random.uniform(0.82, 1.18),
            "order_value_multiplier": detail["order_value_multiplier"] * random.uniform(0.88, 1.12),
            "discount_multiplier": detail["discount_multiplier"] * random.uniform(0.85, 1.15),
            "refund_multiplier": detail["refund_multiplier"] * random.uniform(0.80, 1.20),
            "daily_impressions": (
                detail["daily_impressions"] * random.uniform(0.75, 1.25)
                if detail["daily_impressions"] is not None else None
            ),
            "ctr": (
                detail["ctr"] * random.uniform(0.75, 1.25)
                if detail["ctr"] is not None else None
            ),
            "cpc": (
                detail["cpc"] * random.uniform(0.80, 1.20)
                if detail["cpc"] is not None else None
            ),
        }

campaigns_df = pd.DataFrame(campaigns)
campaign_ids = list(campaign_profiles.keys())
campaign_traffic_weights = [
    campaign_profiles[campaign_id]["traffic_weight"]
    for campaign_id in campaign_ids
]


# ---------------------------------------------------------
# 4. Session dates, visitors, and website sessions
# ---------------------------------------------------------

# A little seasonality and a weekend difference help avoid a flat date pattern.
month_weights = {
    1: 1.03,
    2: 0.96,
    3: 0.98,
    4: 1.00,
    5: 1.02,
    6: 1.04,
    7: 1.03,
    8: 1.05,
    9: 1.07,
    10: 1.12,
    11: 1.28,
    12: 1.32,
}

date_values = []
date_weights = []
for day_number in range((end_date - start_date).days):
    current_day = start_date + timedelta(days=day_number)
    weekend_factor = 0.90 if current_day.weekday() >= 5 else 1.0
    date_values.append(current_day)
    date_weights.append(month_weights[current_day.month] * weekend_factor)


def clamp(value, lower, upper):
    return max(lower, min(value, upper))


def random_session_time():
    session_date = random.choices(date_values, weights=date_weights, k=1)[0]
    seconds_after_midnight = random.randint(8 * 3600, 21 * 3600)
    return session_date + timedelta(seconds=seconds_after_midnight)


visitor_ids = [f"VIS{i:05d}" for i in range(1, number_of_visitors + 1)]

# Different visitors have different tendencies to return and to purchase.
visitor_profiles = {}
for visitor_id in visitor_ids:
    visitor_profiles[visitor_id] = {
        "visit_weight": random.lognormvariate(0, 0.70),
        "purchase_intent": clamp(random.lognormvariate(0, 0.22), 0.60, 1.55),
        "repeat_intent": random.uniform(0.70, 1.35),
        "customer_region": random.choices(
            ["North", "South", "East", "West", "Central"],
            weights=[22, 24, 16, 28, 10],
            k=1,
        )[0],
    }

session_rows = []
sessions_by_visitor = {visitor_id: [] for visitor_id in visitor_ids}


def create_session(visitor_id, session_number):
    session_start_ts = random_session_time()
    campaign_id = random.choices(
        campaign_ids,
        weights=campaign_traffic_weights,
        k=1,
    )[0]
    channel = campaign_profiles[campaign_id]["channel"]

    # Landing pages and devices vary by acquisition channel.
    if channel == "Paid Search":
        landing_page = random.choices(
            ["Home", "Category", "Product", "Campaign"],
            weights=[12, 32, 42, 14],
            k=1,
        )[0]
    elif channel == "Paid Social":
        landing_page = random.choices(
            ["Home", "Category", "Product", "Campaign"],
            weights=[18, 20, 27, 35],
            k=1,
        )[0]
    elif channel == "Email":
        landing_page = random.choices(
            ["Home", "Category", "Product", "Campaign"],
            weights=[8, 22, 40, 30],
            k=1,
        )[0]
    elif channel == "Organic Search":
        landing_page = random.choices(
            ["Home", "Category", "Product", "Campaign"],
            weights=[12, 38, 40, 10],
            k=1,
        )[0]
    elif channel == "Display":
        landing_page = random.choices(
            ["Home", "Category", "Product", "Campaign"],
            weights=[24, 22, 24, 30],
            k=1,
        )[0]
    elif channel == "Referral":
        landing_page = random.choices(
            ["Home", "Category", "Product", "Campaign"],
            weights=[15, 34, 36, 15],
            k=1,
        )[0]
    else:
        landing_page = random.choices(
            ["Home", "Category", "Product", "Campaign"],
            weights=[40, 25, 25, 10],
            k=1,
        )[0]

    device_type = random.choices(
        ["Desktop", "Mobile", "Tablet"],
        weights=[34, 59, 7],
        k=1,
    )[0]

    session = {
        "session_id": f"SES{session_number:06d}",
        "visitor_id": visitor_id,
        "session_start_ts": session_start_ts,
        "session_end_ts": session_start_ts + timedelta(
            minutes=random.randint(30, 60)
        ),
        "campaign_id": campaign_id,
        "device_type": device_type,
        "landing_page": landing_page,
    }
    session_rows.append(session)
    sessions_by_visitor[visitor_id].append(session)


# Every visitor has at least one session; the remaining sessions are allocated
# with higher weights for visitors who are more likely to return.
session_number = 1
for visitor_id in visitor_ids:
    create_session(visitor_id, session_number)
    session_number += 1

remaining_sessions = number_of_sessions - number_of_visitors
visitor_visit_weights = [
    visitor_profiles[visitor_id]["visit_weight"]
    for visitor_id in visitor_ids
]
for _ in range(remaining_sessions):
    visitor_id = random.choices(
        visitor_ids,
        weights=visitor_visit_weights,
        k=1,
    )[0]
    create_session(visitor_id, session_number)
    session_number += 1

for visitor_id in visitor_ids:
    sessions_by_visitor[visitor_id].sort(
        key=lambda session: session["session_start_ts"]
    )

visit_number_by_session = {}
for visitor_id, visitor_sessions in sessions_by_visitor.items():
    for visit_number, session in enumerate(visitor_sessions, start=1):
        visit_number_by_session[session["session_id"]] = visit_number

sessions_df = pd.DataFrame(session_rows)


# ---------------------------------------------------------
# 5. Generate customers, orders, and website events together
# ---------------------------------------------------------

# The tables are generated together so a purchase event, order, customer, and
# order-producing session always refer to one another consistently.
customers = []
orders = []
events = []
visitor_to_customer = {}
last_order_by_customer = {}
order_number = 1


def add_event(session, event_type, event_ts, order_id=None):
    events.append({
        "event_id": f"EVT{len(events) + 1:07d}",
        "session_id": session["session_id"],
        "event_ts": event_ts,
        "event_type": event_type,
        "order_id": order_id,
    })


def create_order(customer_id, session, order_ts, campaign_profile):
    global order_number

    # Mid-range electronics order values, with channel/campaign-level variation.
    base_revenue = random.triangular(25, 250, 105)
    gross_revenue = round(
        clamp(
            base_revenue * campaign_profile["order_value_multiplier"],
            20,
            250,
        ),
        2,
    )

    # Discounts are common enough to analyse, but not applied to every order.
    channel = campaign_profile["channel"]
    discount_probability = {
        "Paid Search": 0.24,
        "Paid Social": 0.43,
        "Display": 0.35,
        "Email": 0.38,
        "Organic Search": 0.18,
        "Referral": 0.20,
        "Direct": 0.17,
    }[channel]
    discount_probability = clamp(
        discount_probability * campaign_profile["discount_multiplier"],
        0.05,
        0.75,
    )

    if random.random() < discount_probability:
        discount_rate = random.uniform(0.03, 0.20)
        discount_amount = round(
            gross_revenue * min(
                discount_rate * campaign_profile["discount_multiplier"],
                0.25,
            ),
            2,
        )
    else:
        discount_amount = 0.0

    # Refund rates vary slightly, and refunds can never exceed the paid amount.
    refund_probability = clamp(
        0.045 * campaign_profile["refund_multiplier"],
        0.01,
        0.12,
    )
    if random.random() < refund_probability:
        amount_after_discount = gross_revenue - discount_amount
        refund_amount = round(
            amount_after_discount * random.uniform(0.10, 0.65),
            2,
        )
    else:
        refund_amount = 0.0

    order = {
        "order_id": f"ORD{order_number:06d}",
        "customer_id": customer_id,
        "session_id": session["session_id"],
        "order_ts": order_ts,
        "gross_revenue": gross_revenue,
        "discount_amount": discount_amount,
        "refund_amount": refund_amount,
    }
    orders.append(order)
    order_number += 1
    last_order_by_customer[customer_id] = order_ts
    return order


# Process sessions in time order so the first order creates the customer and
# later orders can represent repeat purchases.
for session in sorted(session_rows, key=lambda row: row["session_start_ts"]):
    visitor_id = session["visitor_id"]
    profile = campaign_profiles[session["campaign_id"]]
    visitor_profile = visitor_profiles[visitor_id]
    visit_number = visit_number_by_session[session["session_id"]]
    session_start = session["session_start_ts"]
    session_end = session["session_end_ts"]

    seconds_after_start = 0
    number_of_views = random.choices([1, 2, 3, 4], weights=[15, 40, 32, 13], k=1)[0]
    for _ in range(number_of_views):
        seconds_after_start += random.randint(10, 90)
        add_event(
            session,
            "product_view",
            session_start + timedelta(seconds=seconds_after_start),
        )

    # The probability of adding to cart changes by campaign, landing page,
    # device, visitor intent, and visit number.
    landing_cart_factor = {
        "Home": 0.82,
        "Category": 1.08,
        "Product": 1.27,
        "Campaign": 0.98,
    }[session["landing_page"]]
    device_cart_factor = {
        "Desktop": 1.08,
        "Mobile": 0.96,
        "Tablet": 0.94,
    }[session["device_type"]]
    visit_cart_factor = 0.91 if visit_number == 1 else (1.04 if visit_number >= 3 else 1.0)

    cart_probability = clamp(
        0.38
        * profile["cart_multiplier"]
        * landing_cart_factor
        * device_cart_factor
        * (visitor_profile["purchase_intent"] ** 0.35)
        * visit_cart_factor,
        0.08,
        0.80,
    )

    if random.random() < cart_probability:
        seconds_after_start += random.randint(15, 100)
        cart_ts = session_start + timedelta(seconds=seconds_after_start)
        add_event(session, "add_to_cart", cart_ts)

        device_checkout_factor = {
            "Desktop": 1.08,
            "Mobile": 0.93,
            "Tablet": 0.96,
        }[session["device_type"]]
        checkout_probability = clamp(
            0.63
            * profile["checkout_multiplier"]
            * device_checkout_factor
            * (visitor_profile["purchase_intent"] ** 0.12),
            0.25,
            0.92,
        )

        if random.random() < checkout_probability:
            seconds_after_start += random.randint(15, 120)
            checkout_ts = session_start + timedelta(seconds=seconds_after_start)
            add_event(session, "checkout_started", checkout_ts)

            customer_id = visitor_to_customer.get(visitor_id)
            is_repeat_order = customer_id is not None

            if not is_repeat_order:
                purchase_probability = clamp(
                    0.39
                    * profile["purchase_multiplier"]
                    * (visitor_profile["purchase_intent"] ** 0.55)
                    * (1.08 if visit_number >= 2 else 0.92),
                    0.06,
                    0.72,
                )
            else:
                last_order_ts = last_order_by_customer[customer_id]
                days_since_last_order = max(
                    0,
                    (session_start - last_order_ts).days,
                )
                if days_since_last_order < 30:
                    recency_factor = 0.18
                elif days_since_last_order < 90:
                    recency_factor = 0.68
                else:
                    recency_factor = 1.12

                purchase_probability = clamp(
                    0.24
                    * profile["repeat_multiplier"]
                    * visitor_profile["repeat_intent"]
                    * recency_factor,
                    0.01,
                    0.55,
                )

            if random.random() < purchase_probability:
                # Keep the purchase timestamp after checkout and within the session.
                order_ts = max(
                    session_start + timedelta(minutes=random.randint(5, 20)),
                    checkout_ts + timedelta(minutes=random.randint(1, 4)),
                )
                order_ts = min(order_ts, session_end - timedelta(seconds=30))

                if not is_repeat_order:
                    customer_id = f"CUST{len(customers) + 1:05d}"
                    customers.append({
                        "customer_id": customer_id,
                        "visitor_id": visitor_id,
                        "customer_region": visitor_profile["customer_region"],
                    })
                    visitor_to_customer[visitor_id] = customer_id

                order = create_order(
                    customer_id,
                    session,
                    order_ts,
                    profile,
                )
                add_event(session, "purchase", order["order_ts"], order["order_id"])


# Create the dataframes using the existing CSV/table structures.
customers_df = pd.DataFrame(
    customers,
    columns=["customer_id", "visitor_id", "customer_region"],
)
orders_df = pd.DataFrame(
    orders,
    columns=[
        "order_id",
        "customer_id",
        "session_id",
        "order_ts",
        "gross_revenue",
        "discount_amount",
        "refund_amount",
    ],
)
events_df = pd.DataFrame(
    events,
    columns=["event_id", "session_id", "event_ts", "event_type", "order_id"],
)


# ---------------------------------------------------------
# 6. Generate marketing spend
# ---------------------------------------------------------

marketing_spend = []

for day_number in range((end_date - start_date).days):
    current_day = start_date + timedelta(days=day_number)
    weekend_factor = 0.90 if current_day.weekday() >= 5 else 1.0
    seasonal_factor = month_weights[current_day.month] * weekend_factor

    for campaign in campaigns:
        campaign_id = campaign["campaign_id"]
        profile = campaign_profiles[campaign_id]

        # Do not invent spend for untracked organic, referral, and direct campaigns.
        if not campaign["cost_tracking_available"]:
            continue

        baseline_impressions = profile["daily_impressions"] * seasonal_factor
        impressions = int(max(
            100,
            random.gauss(baseline_impressions, baseline_impressions * 0.20),
        ))

        daily_ctr = clamp(
            random.gauss(profile["ctr"], profile["ctr"] * 0.18),
            profile["ctr"] * 0.35,
            profile["ctr"] * 1.65,
        )
        clicks = min(impressions, max(0, int(round(impressions * daily_ctr))))

        if clicks == 0:
            spend_amount = 0.0
        else:
            daily_cpc = max(
                0.05,
                random.gauss(profile["cpc"], profile["cpc"] * 0.15),
            )
            spend_amount = round(clicks * daily_cpc, 2)

        marketing_spend.append({
            "spend_date": current_day.date(),
            "campaign_id": campaign_id,
            "impressions": impressions,
            "clicks": clicks,
            "spend_amount": spend_amount,
        })

marketing_spend_df = pd.DataFrame(
    marketing_spend,
    columns=["spend_date", "campaign_id", "impressions", "clicks", "spend_amount"],
)


# ---------------------------------------------------------
# 7. Validation checks
# ---------------------------------------------------------

purchase_events = events_df[events_df["event_type"] == "purchase"]

assert campaigns_df["campaign_id"].is_unique
assert customers_df["customer_id"].is_unique
assert customers_df["visitor_id"].is_unique
assert sessions_df["session_id"].is_unique
assert events_df["event_id"].is_unique
assert orders_df["order_id"].is_unique

assert sessions_df["campaign_id"].isin(campaigns_df["campaign_id"]).all()
assert events_df["session_id"].isin(sessions_df["session_id"]).all()
assert orders_df["customer_id"].isin(customers_df["customer_id"]).all()
assert orders_df["session_id"].isin(sessions_df["session_id"]).all()
assert orders_df["order_id"].isin(purchase_events["order_id"]).all()
assert set(purchase_events["order_id"]) == set(orders_df["order_id"])
assert len(purchase_events) == len(orders_df)
assert not marketing_spend_df.duplicated(subset=["spend_date", "campaign_id"]).any()
assert (marketing_spend_df["clicks"] <= marketing_spend_df["impressions"]).all()
assert (marketing_spend_df["spend_amount"] >= 0).all()
assert (orders_df["gross_revenue"] >= 0).all()
assert (orders_df["discount_amount"] >= 0).all()
assert (orders_df["refund_amount"] >= 0).all()
assert (
    orders_df["discount_amount"] + orders_df["refund_amount"]
    <= orders_df["gross_revenue"] + 0.01
).all()

sessions_lookup = sessions_df.set_index("session_id")
customers_lookup = customers_df.set_index("customer_id")

# The order's customer must match the visitor tied to the order session.
for order in orders_df.itertuples(index=False):
    session_visitor = sessions_lookup.loc[order.session_id, "visitor_id"]
    customer_visitor = customers_lookup.loc[order.customer_id, "visitor_id"]
    assert session_visitor == customer_visitor
    assert (
        sessions_lookup.loc[order.session_id, "session_start_ts"]
        <= order.order_ts
        <= sessions_lookup.loc[order.session_id, "session_end_ts"]
    )

# Every customer must have at least one order, and each purchase event must link to one order.
assert set(customers_df["customer_id"]) == set(orders_df["customer_id"])
assert sessions_df["session_id"].nunique() == number_of_sessions
assert set(events_df["event_type"].unique()).issubset(
    {"product_view", "add_to_cart", "checkout_started", "purchase"}
)
assert events_df["event_ts"].notna().all()

# Event timestamps must remain inside their session bounds.
event_session_check = events_df.merge(
    sessions_df[["session_id", "session_start_ts", "session_end_ts"]],
    on="session_id",
    how="left",
)
assert (event_session_check["event_ts"] >= event_session_check["session_start_ts"]).all()
assert (event_session_check["event_ts"] <= event_session_check["session_end_ts"]).all()


# ---------------------------------------------------------
# 8. Save all six raw datasets
# ---------------------------------------------------------

campaigns_df.to_csv(raw_dir / "campaigns.csv", index=False)
customers_df.to_csv(raw_dir / "customers.csv", index=False)
sessions_df.to_csv(raw_dir / "website_sessions.csv", index=False)
events_df.to_csv(raw_dir / "website_events.csv", index=False)
orders_df.to_csv(raw_dir / "orders.csv", index=False)
marketing_spend_df.to_csv(raw_dir / "marketing_spend.csv", index=False)


# ---------------------------------------------------------
# 9. Display results
# ---------------------------------------------------------

repeat_customer_count = int((orders_df.groupby("customer_id").size() > 1).sum())
repeat_customer_rate = (
    repeat_customer_count / len(customers_df) * 100 if len(customers_df) else 0
)
session_order_rate = (
    orders_df["session_id"].nunique() / len(sessions_df) * 100
    if len(sessions_df) else 0
)

print("\nRaw data generated successfully!")
print(f"Output folder: {raw_dir}")
print("\nDataset row counts:")
print(f"Campaigns:        {len(campaigns_df):,}")
print(f"Customers:        {len(customers_df):,}")
print(f"Website sessions: {len(sessions_df):,}")
print(f"Website events:   {len(events_df):,}")
print(f"Orders:           {len(orders_df):,}")
print(f"Marketing spend:  {len(marketing_spend_df):,}")
print("\nQuick data checks:")
print(f"Repeat customers: {repeat_customer_count:,} ({repeat_customer_rate:.1f}% of customers)")
print(f"Sessions with an order: {orders_df['session_id'].nunique():,} ({session_order_rate:.2f}% of sessions)")
print("\nAll validation checks passed.")


Raw data folder: C:\Users\mayur\OneDrive\Desktop\jbooks\acquisition-to-retention-analytics\data\raw
Existing CSVs with matching names will be overwritten.

Raw data generated successfully!
Output folder: C:\Users\mayur\OneDrive\Desktop\jbooks\acquisition-to-retention-analytics\data\raw

Dataset row counts:
Campaigns:        19
Customers:        3,302
Website sessions: 45,000
Website events:   144,097
Orders:           4,278
Marketing spend:  6,552

Quick data checks:
Repeat customers: 779 (23.6% of customers)
Sessions with an order: 4,278 (9.51% of sessions)

All validation checks passed.
